In [1]:
import datetime as dt
from pathlib import Path

import ephem
import numpy as np
import pandas as pd

In [2]:
BREEDTE, LENGTE = "51.22", "4.40"
LOKALE_TIJD = dt.timedelta(minutes=17, seconds=30)
WET_1892 = dt.datetime(1892, 5, 1)

ZOMERTIJD = [
    ("1918-03-09 00:00", "1918-10-05 23:00"), ("1919-03-01 23:00", "1919-10-04 23:00"),
    ("1920-02-14 23:00", "1920-10-23 23:00"), ("1921-03-14 23:00", "1921-10-25 23:00"),
    ("1922-03-25 23:00", "1922-10-07 23:00"), ("1923-04-21 23:00", "1923-10-06 23:00"),
    ("1924-03-29 23:00", "1924-10-04 23:00"), ("1925-04-04 23:00", "1925-10-03 23:00"),
    ("1926-04-17 23:00", "1926-10-02 23:00"), ("1927-04-09 23:00", "1927-10-01 23:00"),
    ("1928-04-14 23:00", "1928-10-07 02:00"), ("1929-04-21 02:00", "1929-10-06 02:00"),
    ("1930-04-13 02:00", "1930-10-05 02:00"), ("1931-04-19 02:00", "1931-10-04 02:00"),
    ("1932-04-03 02:00", "1932-10-02 02:00"), ("1933-03-26 02:00", "1933-10-08 02:00"),
    ("1934-04-08 02:00", "1934-10-07 02:00"), ("1935-03-31 02:00", "1935-10-06 02:00"),
    ("1936-04-19 02:00", "1936-10-04 02:00"), ("1937-04-04 02:00", "1937-10-03 02:00"),
    ("1938-03-27 02:00", "1938-10-02 02:00"), ("1939-04-16 02:00", "1939-11-19 02:00"),
    ("1940-02-25 02:00", "1940-12-31 23:00"),
]
BEZETTING = []

_lees = lambda p: [
    (dt.datetime.strptime(a, "%Y-%m-%d %H:%M"), dt.datetime.strptime(b, "%Y-%m-%d %H:%M"))
    for a, b in p
]
ZOMERTIJD, BEZETTING = _lees(ZOMERTIJD), _lees(BEZETTING)


def offset(moment):
    """Wettelijke tijd in Antwerpen min UTC."""
    if moment < WET_1892:
        return LOKALE_TIJD
    for start, einde in BEZETTING:
        if start <= moment < einde:
            return dt.timedelta(hours=2)
    for start, einde in ZOMERTIJD:
        if start <= moment < einde:
            return dt.timedelta(hours=1)
    return dt.timedelta(0)

In [3]:
def grenzen(datum):
    """Ochtendschemer, zonsopgang, zonsondergang en avondschemer in UTC."""
    obs = ephem.Observer()
    obs.lat, obs.lon = BREEDTE, LENGTE
    obs.pressure = 0
    zon = ephem.Sun()

    obs.date = datum.isoformat()
    obs.horizon = "-6"
    schemer_start = obs.next_rising(zon, use_center=True).datetime()
    schemer_eind = obs.next_setting(zon, use_center=True).datetime()

    obs.date = datum.isoformat()
    obs.horizon = "-0:34"
    opgang = obs.next_rising(zon).datetime()
    ondergang = obs.next_setting(zon).datetime()

    return schemer_start, opgang, ondergang, schemer_eind


def classificeer(moment_utc, g):
    schemer_start, opgang, ondergang, schemer_eind = g
    if moment_utc < schemer_start:
        return "nacht"
    if moment_utc < opgang:
        return "ochtendschemering"
    if moment_utc < ondergang:
        return "dag"
    if moment_utc < schemer_eind:
        return "avondschemering"
    return "nacht"


def zonshoogte(moment_utc):
    """Hoogte van het middelpunt van de zon in graden."""
    obs = ephem.Observer()
    obs.lat, obs.lon = BREEDTE, LENGTE
    obs.pressure = 0
    obs.date = moment_utc
    zon = ephem.Sun()
    zon.compute(obs)
    return float(zon.alt) * 180 / ephem.pi

In [28]:
MAP_IN = Path("../../../data/algemeen/gebeurtenisdata_ruw")
MAP_UIT = Path("../../../data/algemeen/gebeurtenisdata_astro")
BESTANDSPATROON = "*.xlsx"

DATUM_KOLOM = "E_DATE_clean"
TIJD_KOLOM = "E_TIME_clean"
RUWE_TIJD_KOLOM = "E_TIME"
OUDE_KOLOM = "astronomical_category"

EERSTE_JAAR, LAATSTE_JAAR = 1876, 1939

ACHTERVOEGSELS = ["_with_astronomical_categories", "_with_text", "_astrotime"]


def bronnaam(pad):
    """Bestandsnaam zonder de achtervoegsels van eerdere verwerkingsstappen."""
    naam = pad.stem
    for staart in ACHTERVOEGSELS:
        if naam.endswith(staart):
            naam = naam[: -len(staart)]
    return naam


# Bewust uitgesloten uit het corpus.
UITGESLOTEN = {
    # Bezettingsjaren: politionele registratie onder Duits bestuur is niet
    # vergelijkbaar met de rest van de periode.
    "MA#30828", "1700#1", "MA#17876", "MA#17877", "1700#12",
    "1699#125", "1699#124", "MA#17878", "MA#17879",
    # Bureel Linkeroever, valt buiten het onderzochte stedelijke gebied.
    "MA#41579",
}

DAGDEEL_VAN_STANDAARDTIJD = {
    "09:00": "ochtend", "12:00": "middag", "15:00": "namiddag",
    "21:00": "avond", "00:00": "nacht",
}
DAGDEEL_INTERVAL = {
    "ochtend": (6, 12), "middag": (11, 14), "namiddag": (12, 18),
    "avond": (18, 24), "nacht": (0, 6),
}

if not MAP_IN.exists():
    raise FileNotFoundError(f"invoermap bestaat niet: {MAP_IN}")
if MAP_UIT.resolve() == MAP_IN.resolve():
    raise ValueError("uitvoermap mag niet gelijk zijn aan de invoermap")

print(f"invoer:  {MAP_IN}")
print(f"uitvoer: {MAP_UIT}")

invoer:  /Volumes/Extreme SSD/doctoraat/gebeurtenisboeken/ANALYSE/0. dataframes
uitvoer: /Volumes/Extreme SSD/doctoraat/gebeurtenisboeken/ANALYSE/0. dataframes_astro_corr_0826


In [15]:
for p in sorted(MAP_IN.glob("*.xlsx"))[:5]:
    print(f"{p.stem}  ->  {bronnaam(p)}")

._450#58_astrotime  ->  ._450#58
._MA#30817_astrotime  ->  ._MA#30817
._MA#30818_astrotime  ->  ._MA#30818
._MA#30819_astrotime  ->  ._MA#30819
._MA#30820_astrotime  ->  ._MA#30820


In [30]:
bestanden = sorted(MAP_IN.glob(BESTANDSPATROON))
bestanden = [
    p for p in bestanden
    if not p.name.startswith(("~$", "._")) and p.stat().st_size > 0
]
uitgesloten_gevonden = [p for p in bestanden if bronnaam(p) in UITGESLOTEN]
bestanden = [p for p in bestanden if bronnaam(p) not in UITGESLOTEN]
MAP_UIT.mkdir(parents=True, exist_ok=True)

if not bestanden:
    raise FileNotFoundError(f"geen bestanden gevonden met {BESTANDSPATROON} in {MAP_IN}")

print(f"{len(uitgesloten_gevonden)} van de {len(UITGESLOTEN)} uitgesloten boeken gevonden en overgeslagen")

rijen, onleesbaar = [], []
for pad in bestanden:
    try:
        kolommen = pd.read_excel(pad, nrows=0).columns
        datum = pd.to_datetime(
            pd.read_excel(pad, usecols=[DATUM_KOLOM])[DATUM_KOLOM],
            format="%d/%m/%Y", errors="coerce",
        )
    except Exception as fout:
        onleesbaar.append((pad.name, type(fout).__name__))
        continue
    rijen.append({
        "bron": bronnaam(pad),
        "datum_kolom": DATUM_KOLOM in kolommen,
        "tijd_kolom": TIJD_KOLOM in kolommen,
        "ruwe_tijd_kolom": RUWE_TIJD_KOLOM in kolommen,
        "in_periode": int(datum.dt.year.between(EERSTE_JAAR, LAATSTE_JAAR).sum()),
    })

inventaris = pd.DataFrame(rijen)

buiten = inventaris[inventaris["in_periode"] == 0]
inventaris = inventaris[inventaris["in_periode"] > 0].reset_index(drop=True)
bestanden = [p for p in bestanden if bronnaam(p) in set(inventaris["bron"])]

print(f"{len(buiten)} boeken volledig na {LAATSTE_JAAR}, overgeslagen")
print(f"{len(bestanden)} gebeurtenisboeken in het corpus")
print(inventaris[["datum_kolom", "tijd_kolom", "ruwe_tijd_kolom"]].sum().to_string())

10 van de 10 uitgesloten boeken gevonden en overgeslagen
63 boeken volledig na 1939, overgeslagen
197 gebeurtenisboeken in het corpus
datum_kolom        197
tijd_kolom         197
ruwe_tijd_kolom    197


In [32]:
alle_datums = pd.date_range(f"{EERSTE_JAAR}-01-01", f"{LAATSTE_JAAR}-12-31").date
GRENZEN = {d: grenzen(d) for d in alle_datums}
print(f"zonnestand berekend voor {len(GRENZEN)} datums ({EERSTE_JAAR}-{LAATSTE_JAAR})")

zonnetabel = pd.DataFrame(
    [(d, *GRENZEN[d]) for d in alle_datums],
    columns=["datum", "ochtendschemer", "zonsopgang", "zonsondergang", "avondschemer"],
)
zonnetabel.to_parquet("../../../data/H4/normalisering/zonnestand_antwerpen.parquet", index=False)

zonnestand berekend voor 23375 datums (1876-1939)


In [33]:
def interval_categorie(rij_datum, dagdeel):
    """Categorie voor een record waarvan alleen het dagdeel bekend is."""
    van, tot = DAGDEEL_INTERVAL[dagdeel]
    g = GRENZEN[rij_datum]
    basis = dt.datetime.combine(rij_datum, dt.time())
    gevonden = set()
    for minuut in range(van * 60, tot * 60 + 1, 15):
        moment = basis + dt.timedelta(minutes=minuut)
        gevonden.add(classificeer(moment - offset(moment), g))
    return gevonden.pop() if len(gevonden) == 1 else "onbepaald"


CATEGORIEEN = ["dag", "nacht", "ochtendschemering", "avondschemering"]


def verwerk(pad):
    df = pd.read_excel(pad)
    df.insert(0, "bron", bronnaam(pad))

    datum = pd.to_datetime(df[DATUM_KOLOM], format="%d/%m/%Y", errors="coerce")
    tijd = pd.to_datetime(df[TIJD_KOLOM], format="%H:%M", errors="coerce")
    klok = datum + (tijd - tijd.dt.normalize())
    buiten_bereik = datum.notna() & ~datum.dt.year.between(EERSTE_JAAR, LAATSTE_JAAR)
    bruikbaar = klok.notna() & ~buiten_bereik

    # Een tijdstip geldt als ingevuld wanneer het een standaardwaarde is en de ruwe
    # NER-waarde geen enkel cijfer bevat.
    is_standaard = df[TIJD_KOLOM].isin(DAGDEEL_VAN_STANDAARDTIJD)
    if RUWE_TIJD_KOLOM in df.columns:
        geen_cijfer = ~df[RUWE_TIJD_KOLOM].astype(str).str.contains(r"\d", na=False)
    else:
        geen_cijfer = pd.Series(True, index=df.index)
    ingevuld = bruikbaar & is_standaard & geen_cijfer
    afwijkend = int((bruikbaar & geen_cijfer & ~is_standaard).sum())

    df["tijd_precisie"] = np.where(~bruikbaar, "onbekend", np.where(ingevuld, "dagdeel", "exact"))
    df["dagdeel"] = df[TIJD_KOLOM].map(DAGDEEL_VAN_STANDAARDTIJD).where(ingevuld)

    # De categorie uit de vorige verwerking, om het verschil te kunnen rapporteren.
    if OUDE_KOLOM in df.columns:
        df = df.rename(columns={OUDE_KOLOM: "astro_oud"})
        df["astro_oud"] = df["astro_oud"].where(df["astro_oud"].isin(CATEGORIEEN))
    else:
        df["astro_oud"] = pd.NA

    def toewijzen(moment, corrigeren):
        g = GRENZEN[moment.date()]
        moment = moment.to_pydatetime()
        if corrigeren:
            moment = moment - offset(moment)
        return classificeer(moment, g)

    df["astro_ongecorrigeerd"] = pd.NA
    df["astro"] = pd.NA
    df.loc[bruikbaar, "astro_ongecorrigeerd"] = klok[bruikbaar].map(lambda m: toewijzen(m, False))
    df.loc[bruikbaar, "astro"] = klok[bruikbaar].map(lambda m: toewijzen(m, True))

    df["astro_verandert"] = (
        (df["astro_ongecorrigeerd"] != df["astro"]).where(bruikbaar).astype("boolean")
    )
    heeft_oud = bruikbaar & df["astro_oud"].notna()
    df["verschilt_van_oud"] = (
        (df["astro_oud"] != df["astro"]).where(heeft_oud).astype("boolean")
    )

    utc = klok[bruikbaar].map(lambda m: m.to_pydatetime() - offset(m.to_pydatetime()))
    minuten = lambda a, b: (a - b).total_seconds() / 60
    df["zonshoogte"] = pd.NA
    df["min_na_zonsondergang"] = pd.NA
    df["min_voor_zonsopgang"] = pd.NA
    df["daglengte_uren"] = pd.NA
    df.loc[bruikbaar, "zonshoogte"] = utc.map(zonshoogte).round(2)
    df.loc[bruikbaar, "min_na_zonsondergang"] = [
        round(minuten(m, GRENZEN[m.date()][2])) for m in utc
    ]
    df.loc[bruikbaar, "min_voor_zonsopgang"] = [
        round(minuten(GRENZEN[m.date()][1], m)) for m in utc
    ]
    df.loc[bruikbaar, "daglengte_uren"] = [
        round(minuten(GRENZEN[d][2], GRENZEN[d][1]) / 60, 2) for d in datum[bruikbaar].dt.date
    ]

    df["astro_interval"] = df["astro"]
    for i in df.index[ingevuld]:
        df.at[i, "astro_interval"] = interval_categorie(datum[i].date(), df.at[i, "dagdeel"])

    df["_datum"] = datum
    return df, {
        "bron": bronnaam(pad),
        "records": len(df),
        "bruikbaar": int(bruikbaar.sum()),
        "buiten_bereik": int(buiten_bereik.sum()),
        "dagdeel": int(ingevuld.sum()),
        "afwijkend": afwijkend,
        "gewijzigd": int(df["astro_verandert"].fillna(False).sum()),
        "oud_bekend": int(heeft_oud.sum()),
        "verschilt_van_oud": int(df["verschilt_van_oud"].fillna(False).sum()),
        "eerste": datum.min(),
        "laatste": datum.max(),
    }

In [34]:
bestanden_run = bestanden

DIAGNOSE_KOLOMMEN = [
    "bron", "tijd_precisie", "dagdeel", "astro_oud", "astro_ongecorrigeerd", "astro",
    "astro_interval", "astro_verandert", "verschilt_van_oud", "zonshoogte",
    "min_na_zonsondergang", "min_voor_zonsopgang", "daglengte_uren", "_datum",
]

delen, overzicht = [], []
for n, pad in enumerate(bestanden_run, 1):
    try:
        deel, samenvatting = verwerk(pad)
    except Exception as fout:
        print(f"  overgeslagen: {pad.name} -> {type(fout).__name__}: {fout}")
        continue
    deel.drop(columns="_datum").to_excel(MAP_UIT / f"{bronnaam(pad)}.xlsx", index=False)
    delen.append(deel[DIAGNOSE_KOLOMMEN])
    overzicht.append(samenvatting)
    if n % 25 == 0:
        print(f"  {n}/{len(bestanden_run)} verwerkt")

overzicht = pd.DataFrame(overzicht)
corpus = pd.concat(delen, ignore_index=True)
print(f"\n{len(corpus)} records uit {len(delen)} bestanden")
print(overzicht.to_string(index=False))

  25/197 verwerkt
  50/197 verwerkt
  75/197 verwerkt
  100/197 verwerkt
  125/197 verwerkt
  150/197 verwerkt
  175/197 verwerkt

383125 records uit 197 bestanden
    bron  records  bruikbaar  buiten_bereik  dagdeel  afwijkend  gewijzigd  oud_bekend  verschilt_van_oud     eerste    laatste
  1253#4     1255        340            336        4          0         21           0                  0 1939-02-28 1940-11-27
1699#126     4645       1181              0       61          0         76           0                  0 1919-01-01 1920-12-17
 1700#10     3995       1343              0      145          0          0           0                  0 1911-01-09 1912-12-30
 1700#11     4517       1450              0      116          0          0           0                  0 1910-01-02 1910-12-13
 1700#13     4142       1284              0      130          0          0           0                  0 1912-01-02 1912-12-31
 1700#15     4348       1420              0      193          0     